# XGBoost Baseline: NJ Housing Price Prediction

Trains an XGBoost regressor on the same NJ housing data used for QLoRA fine-tuning,
then compares both models side-by-side.

## What this notebook does
1. Loads `rajkumar4466/nj-housing-prices` and parses 7 tabular features from prompt text
2. Pushes tabular version to HuggingFace as `rajkumar4466/nj-housing-prices-tabular`
3. Trains `XGBRegressor` with hyperparameter tuning via `GridSearchCV`
4. Evaluates on held-out test set: MAE, RMSE, R², MAPE
5. Generates predicted vs actual scatter plot and feature importance chart
6. Saves model, metrics, and comparison table to `models/`
7. Pushes both models to HuggingFace with model cards and comparison artifacts

## Requirements
- `pip install -r requirements.txt` (includes xgboost)
- HuggingFace token for dataset/model push (`huggingface-cli login`)
- No GPU needed — runs locally in under a minute

In [ ]:
import os
import re
import json
import time

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

_nb_dir = os.getcwd()
repo_root = _nb_dir if os.path.isdir(os.path.join(_nb_dir, "lambda")) else os.path.dirname(_nb_dir)
MODELS_DIR = os.path.join(repo_root, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

_start_time = time.time()
print(f"Repo root: {repo_root}")
print(f"Models dir: {MODELS_DIR}")

## Step 1: Create Tabular Dataset from Prompt Text

The existing HuggingFace dataset has `prompt` + `price` columns. The prompt format
(from `lambda/prompt_utils.py`) is:

```
Property: {property_type} in zip {zip_code}. {bedrooms} bedrooms, {bathrooms} bathrooms, {sqft} sqft living area, {lot_size:.2f} acre lot, built in {year_built}. Predicted price: $
```

We parse all 7 features back out using regex.

In [ ]:
HF_DATASET = "rajkumar4466/nj-housing-prices"
HF_TABULAR = "rajkumar4466/nj-housing-prices-tabular"

print(f"Loading dataset from HuggingFace: {HF_DATASET}...")
ds = load_dataset(HF_DATASET)
print(f"Splits: {list(ds.keys())}")
for split_name, split_data in ds.items():
    print(f"  {split_name}: {len(split_data):,} records")

# Regex to parse all 7 features from the prompt
PROMPT_PATTERN = re.compile(
    r"Property:\s+(.+?)\s+in\s+zip\s+(\d{5})\.\s+"
    r"(\d+)\s+bedrooms,\s+([\d.]+)\s+bathrooms,\s+"
    r"(\d+)\s+sqft\s+living\s+area,\s+"
    r"([\d.]+)\s+acre\s+lot,\s+built\s+in\s+(\d{4})"
)


def parse_prompt(prompt: str) -> dict:
    """Parse 7 features from a formatted prompt string."""
    m = PROMPT_PATTERN.search(prompt)
    if not m:
        raise ValueError(f"Could not parse prompt: {prompt[:80]}")
    return {
        "property_type": m.group(1),
        "zip_code": m.group(2),
        "bedrooms": int(m.group(3)),
        "bathrooms": float(m.group(4)),
        "sqft": int(m.group(5)),
        "lot_size": float(m.group(6)),
        "year_built": int(m.group(7)),
    }


# Quick test on first record
sample = ds["train"][0]
parsed = parse_prompt(sample["prompt"])
print(f"\nSample prompt: {sample['prompt'][:80]}...")
print(f"Parsed features: {parsed}")
print(f"Price: ${sample['price']:,.0f}")

In [ ]:
def split_to_tabular(split_data) -> pd.DataFrame:
    """Convert a HuggingFace split (prompt+price) to tabular DataFrame."""
    rows = []
    for record in split_data:
        features = parse_prompt(record["prompt"])
        features["price"] = record["price"]
        rows.append(features)
    return pd.DataFrame(rows)


df_train = split_to_tabular(ds["train"])
df_val = split_to_tabular(ds["validation"])
df_test = split_to_tabular(ds["test"])

print(f"Tabular datasets created:")
print(f"  Train: {len(df_train):,} rows")
print(f"  Val:   {len(df_val):,} rows")
print(f"  Test:  {len(df_test):,} rows")
print(f"\nColumns: {list(df_train.columns)}")
print(f"\nSample row:")
print(df_train.iloc[0].to_dict())

In [ ]:
# Push tabular dataset to HuggingFace
tabular_ds = DatasetDict({
    "train": Dataset.from_pandas(df_train, preserve_index=False),
    "validation": Dataset.from_pandas(df_val, preserve_index=False),
    "test": Dataset.from_pandas(df_test, preserve_index=False),
})

print(f"Tabular DatasetDict:")
print(tabular_ds)

try:
    tabular_ds.push_to_hub(HF_TABULAR, private=False)
    print(f"\nDataset pushed to: https://huggingface.co/datasets/{HF_TABULAR}")
except Exception as e:
    print(f"\nWARNING: Could not push to Hub: {e}")
    print("Run 'huggingface-cli login' or set HF_TOKEN env var, then rerun this cell.")
    print(f"Target repo: {HF_TABULAR}")

## Step 2: Train XGBoost Model

- `property_type` → one-hot encoded
- `zip_code` → integer (ordinal; zip codes carry geographic signal)
- Hyperparameter tuning via `GridSearchCV` on validation set

In [ ]:
def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    """Encode features for XGBoost: one-hot property_type, integer zip_code."""
    df = df.copy()
    # zip_code as integer (preserves geographic ordering)
    df["zip_code"] = df["zip_code"].astype(int)
    # One-hot encode property_type
    df = pd.get_dummies(df, columns=["property_type"], prefix="pt", dtype=int)
    return df


df_train_enc = prepare_features(df_train)
df_val_enc = prepare_features(df_val)
df_test_enc = prepare_features(df_test)

# Align columns across splits (in case a property type is missing from a split)
all_cols = sorted(set(df_train_enc.columns) | set(df_val_enc.columns) | set(df_test_enc.columns))
for df in [df_train_enc, df_val_enc, df_test_enc]:
    for col in all_cols:
        if col not in df.columns:
            df[col] = 0

feature_cols = [c for c in sorted(df_train_enc.columns) if c != "price"]

X_train = df_train_enc[feature_cols]
y_train = df_train_enc["price"]
X_val = df_val_enc[feature_cols]
y_val = df_val_enc["price"]
X_test = df_test_enc[feature_cols]
y_test = df_test_enc["price"]

print(f"Feature columns ({len(feature_cols)}): {feature_cols}")
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

In [ ]:
# Combine train + val for GridSearchCV (it does its own CV splits)
X_trainval = pd.concat([X_train, X_val], ignore_index=True)
y_trainval = pd.concat([y_train, y_val], ignore_index=True)

param_grid = {
    "n_estimators": [200, 500, 1000],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
}

print(f"Running GridSearchCV with {np.prod([len(v) for v in param_grid.values()])} param combos x 3 folds...")
train_start = time.time()

xgb_base = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

grid_search = GridSearchCV(
    xgb_base,
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    verbose=1,
    n_jobs=-1,
)

grid_search.fit(X_trainval, y_trainval)

train_elapsed = time.time() - train_start
print(f"\nGridSearchCV complete in {train_elapsed:.1f}s")
print(f"Best params: {grid_search.best_params_}")
print(f"Best CV MAE: ${-grid_search.best_score_:,.0f}")

model = grid_search.best_estimator_

## Step 3: Evaluate on Held-Out Test Set

Same 4 metrics as `03_evaluate.ipynb`: MAE, RMSE, R², MAPE

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test.values - y_pred) / y_test.values)) * 100

print("=" * 60)
print("XGBOOST TEST SET METRICS")
print("=" * 60)
print(f"Test set size: {len(y_test):,} records")
print(f"")
print(f"MAE:  ${mae:,.0f}")
print(f"RMSE: ${rmse:,.0f}")
print(f"R²:   {r2:.4f}")
print(f"MAPE: {mape:.1f}%")
print("=" * 60)

## Step 4: Visualizations

In [ ]:
# Predicted vs Actual scatter plot (same style as 03_evaluate.ipynb)
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(y_test, y_pred, alpha=0.3, s=10, color="steelblue", label="Predictions")

lims = [
    min(y_test.min(), y_pred.min()),
    max(y_test.max(), y_pred.max()),
]
ax.plot(lims, lims, "r--", label="Perfect prediction", linewidth=1.5)

ax.set_xlabel("Actual Price ($)", fontsize=12)
ax.set_ylabel("Predicted Price ($)", fontsize=12)
ax.set_title("Predicted vs. Actual Housing Prices\nXGBoost Baseline", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

currency_fmt = FuncFormatter(lambda x, p: f"${x:,.0f}")
ax.xaxis.set_major_formatter(currency_fmt)
ax.yaxis.set_major_formatter(currency_fmt)

plt.tight_layout()
scatter_path = os.path.join(MODELS_DIR, "xgboost_predicted_vs_actual.png")
plt.savefig(scatter_path, dpi=150)
plt.show()
plt.close()
print(f"Scatter plot saved to: {scatter_path}")

In [ ]:
# Feature importance bar chart
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Feature Importance (gain)", fontsize=12)
ax.set_title("XGBoost Feature Importances", fontsize=14)
ax.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
importance_path = os.path.join(MODELS_DIR, "xgboost_feature_importance.png")
plt.savefig(importance_path, dpi=150)
plt.show()
plt.close()
print(f"Feature importance chart saved to: {importance_path}")

In [ ]:
# Side-by-side comparison: QLoRA vs XGBoost
qlora_metrics = {
    "MAE": 140141,
    "RMSE": 190172,
    "R2": 0.6359,
    "MAPE": 23.0,
}

xgboost_metrics = {
    "MAE": float(mae),
    "RMSE": float(rmse),
    "R2": float(r2),
    "MAPE": float(mape),
}

comparison = {
    "qlora": qlora_metrics,
    "xgboost": xgboost_metrics,
}

# Print formatted table
print("=" * 60)
print("MODEL COMPARISON: QLoRA vs XGBoost")
print("=" * 60)
print(f"{'Metric':<10} {'QLoRA':>15} {'XGBoost':>15} {'Winner':>10}")
print("-" * 50)
for metric in ["MAE", "RMSE", "R2", "MAPE"]:
    q_val = qlora_metrics[metric]
    x_val = xgboost_metrics[metric]
    if metric == "R2":
        q_str = f"{q_val:.4f}"
        x_str = f"{x_val:.4f}"
        winner = "XGBoost" if x_val > q_val else "QLoRA"
    elif metric == "MAPE":
        q_str = f"{q_val:.1f}%"
        x_str = f"{x_val:.1f}%"
        winner = "XGBoost" if x_val < q_val else "QLoRA"
    else:
        q_str = f"${q_val:,.0f}"
        x_str = f"${x_val:,.0f}"
        winner = "XGBoost" if x_val < q_val else "QLoRA"
    print(f"{metric:<10} {q_str:>15} {x_str:>15} {winner:>10}")
print("=" * 60)

## Step 5: Save Artifacts

In [ ]:
# Save XGBoost model in native JSON format
model_path = os.path.join(MODELS_DIR, "xgboost_baseline.json")
model.save_model(model_path)
print(f"Model saved to: {model_path}")

# Save XGBoost metrics
metrics_path = os.path.join(MODELS_DIR, "xgboost_metrics.json")
xgb_metrics_full = {
    "model": "XGBRegressor",
    "best_params": grid_search.best_params_,
    "test_set_size": len(y_test),
    "mae": float(mae),
    "rmse": float(rmse),
    "r2": float(r2),
    "mape": float(mape),
    "train_time_seconds": train_elapsed,
}
with open(metrics_path, "w") as f:
    json.dump(xgb_metrics_full, f, indent=2)
print(f"Metrics saved to: {metrics_path}")

# Save comparison table
comparison_path = os.path.join(MODELS_DIR, "comparison.json")
with open(comparison_path, "w") as f:
    json.dump(comparison, f, indent=2)
print(f"Comparison saved to: {comparison_path}")

## Step 6: Push Both Models to HuggingFace

Push the XGBoost model and the QLoRA ONNX model to HuggingFace model repos,
along with metrics and comparison artifacts.

- `rajkumar4466/nj-housing-xgboost-baseline` — XGBoost model + metrics
- `rajkumar4466/nj-housing-qlora-onnx` — ONNX model + tokenizer + metrics

Requires `huggingface-cli login` or `HF_TOKEN` env var.

In [ ]:
from huggingface_hub import HfApi, ModelCard

api = HfApi()
HF_USERNAME = "rajkumar4466"

# --- Push XGBoost model ---
XGB_REPO = f"{HF_USERNAME}/nj-housing-xgboost-baseline"

print(f"Creating/updating repo: {XGB_REPO}")
api.create_repo(repo_id=XGB_REPO, repo_type="model", exist_ok=True)

# Upload model, metrics, and comparison
api.upload_file(
    path_or_fileobj=os.path.join(MODELS_DIR, "xgboost_baseline.json"),
    path_in_repo="xgboost_baseline.json",
    repo_id=XGB_REPO,
    repo_type="model",
)
api.upload_file(
    path_or_fileobj=os.path.join(MODELS_DIR, "xgboost_metrics.json"),
    path_in_repo="xgboost_metrics.json",
    repo_id=XGB_REPO,
    repo_type="model",
)
api.upload_file(
    path_or_fileobj=os.path.join(MODELS_DIR, "comparison.json"),
    path_in_repo="comparison.json",
    repo_id=XGB_REPO,
    repo_type="model",
)
api.upload_file(
    path_or_fileobj=os.path.join(MODELS_DIR, "xgboost_predicted_vs_actual.png"),
    path_in_repo="xgboost_predicted_vs_actual.png",
    repo_id=XGB_REPO,
    repo_type="model",
)
api.upload_file(
    path_or_fileobj=os.path.join(MODELS_DIR, "xgboost_feature_importance.png"),
    path_in_repo="xgboost_feature_importance.png",
    repo_id=XGB_REPO,
    repo_type="model",
)

# Model card for XGBoost
xgb_card_content = f"""---
library_name: xgboost
tags:
  - tabular-regression
  - housing-prices
  - xgboost
  - new-jersey
datasets:
  - rajkumar4466/nj-housing-prices-tabular
metrics:
  - mae
  - rmse
  - r2
  - mape
---

# XGBoost Baseline — NJ Housing Price Prediction

XGBoost regressor trained on 7 structured features from NJ housing data.
Serves as a baseline comparison against a QLoRA fine-tuned Qwen2.5-0.5B LLM.

## Metrics (held-out test set, n={len(y_test):,})

| Metric | XGBoost | QLoRA (Qwen2.5-0.5B) |
|--------|---------|----------------------|
| MAE    | ${mae:,.0f} | $140,141 |
| RMSE   | ${rmse:,.0f} | $190,172 |
| R²     | {r2:.4f} | 0.6359 |
| MAPE   | {mape:.1f}% | 23.0% |

## Best Hyperparameters

```json
{json.dumps(grid_search.best_params_, indent=2)}
```

## Features

| Feature | Type |
|---------|------|
| bedrooms | int |
| bathrooms | float |
| sqft | int |
| lot_size | float |
| year_built | int |
| zip_code | int (ordinal) |
| property_type | one-hot encoded |

## Usage

```python
from xgboost import XGBRegressor
from huggingface_hub import hf_hub_download

path = hf_hub_download("{XGB_REPO}", "xgboost_baseline.json")
model = XGBRegressor()
model.load_model(path)

# Predict (features must be encoded the same way as training)
# model.predict(X)
```

## Dataset

Trained on [rajkumar4466/nj-housing-prices-tabular](https://huggingface.co/datasets/rajkumar4466/nj-housing-prices-tabular)
"""

api.upload_file(
    path_or_fileobj=xgb_card_content.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=XGB_REPO,
    repo_type="model",
)

print(f"XGBoost model pushed to: https://huggingface.co/{XGB_REPO}")

In [ ]:
# --- Push QLoRA ONNX model ---
QLORA_REPO = f"{HF_USERNAME}/nj-housing-qlora-onnx"
ONNX_DIR = os.path.join(repo_root, "lambda", "model_artifacts")

print(f"\nCreating/updating repo: {QLORA_REPO}")
api.create_repo(repo_id=QLORA_REPO, repo_type="model", exist_ok=True)

# Upload all model artifacts (ONNX model + tokenizer files)
onnx_files = [
    f for f in os.listdir(ONNX_DIR)
    if not f.startswith(".") and os.path.isfile(os.path.join(ONNX_DIR, f))
]
print(f"Uploading {len(onnx_files)} files from {ONNX_DIR}...")

for fname in sorted(onnx_files):
    fpath = os.path.join(ONNX_DIR, fname)
    fsize = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  Uploading {fname} ({fsize:.1f} MB)...")
    api.upload_file(
        path_or_fileobj=fpath,
        path_in_repo=fname,
        repo_id=QLORA_REPO,
        repo_type="model",
    )

# Upload comparison to QLoRA repo too
api.upload_file(
    path_or_fileobj=os.path.join(MODELS_DIR, "comparison.json"),
    path_in_repo="comparison.json",
    repo_id=QLORA_REPO,
    repo_type="model",
)

# Model card for QLoRA ONNX
qlora_card_content = f"""---
library_name: onnxruntime
tags:
  - text-regression
  - housing-prices
  - qlora
  - onnx
  - qwen2.5
  - new-jersey
datasets:
  - rajkumar4466/nj-housing-prices
metrics:
  - mae
  - rmse
  - r2
  - mape
---

# QLoRA Fine-tuned Qwen2.5-0.5B (ONNX) — NJ Housing Price Prediction

Qwen2.5-0.5B fine-tuned with QLoRA (4-bit NF4) on NJ housing data, then merged
and exported to ONNX format for CPU inference.

## Metrics (held-out test set)

| Metric | QLoRA (this model) | XGBoost Baseline |
|--------|--------------------|------------------|
| MAE    | $140,141 | ${mae:,.0f} |
| RMSE   | $190,172 | ${rmse:,.0f} |
| R²     | 0.6359 | {r2:.4f} |
| MAPE   | 23.0% | {mape:.1f}% |

## Model Details

- **Base model**: Qwen/Qwen2.5-0.5B
- **Fine-tuning**: QLoRA (rank=16, alpha=32, 4-bit NF4 quantization)
- **Export**: Merged to fp32, exported to ONNX
- **Inference**: ONNX Runtime (CPU), no GPU required

## Prompt Format

```
Property: Single Family in zip 07650. 3 bedrooms, 2.0 bathrooms, 1500 sqft living area, 0.25 acre lot, built in 1990. Predicted price: $
```

The model generates the price value after the `$` prefix.

## Usage

```python
import onnxruntime as ort
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download

path = snapshot_download("{QLORA_REPO}")
session = ort.InferenceSession(f"{{path}}/model.onnx")
tokenizer = AutoTokenizer.from_pretrained(path)
```

## Dataset

Trained on [rajkumar4466/nj-housing-prices](https://huggingface.co/datasets/rajkumar4466/nj-housing-prices)

## Comparison

See [rajkumar4466/nj-housing-xgboost-baseline](https://huggingface.co/{XGB_REPO}) for the XGBoost baseline comparison.
"""

api.upload_file(
    path_or_fileobj=qlora_card_content.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=QLORA_REPO,
    repo_type="model",
)

print(f"QLoRA ONNX model pushed to: https://huggingface.co/{QLORA_REPO}")
print(f"\nBoth models pushed successfully!")
print(f"  XGBoost:    https://huggingface.co/{XGB_REPO}")
print(f"  QLoRA ONNX: https://huggingface.co/{QLORA_REPO}")

In [ ]:
total_elapsed = time.time() - _start_time

print("=" * 60)
print("PHASE 6 SUMMARY")
print("=" * 60)
print(f"XGBoost best params: {grid_search.best_params_}")
print(f"")
print(f"XGBoost Test Metrics:")
print(f"  MAE:  ${mae:,.0f}")
print(f"  RMSE: ${rmse:,.0f}")
print(f"  R²:   {r2:.4f}")
print(f"  MAPE: {mape:.1f}%")
print(f"")
print(f"Artifacts saved to {MODELS_DIR}:")
print(f"  - xgboost_baseline.json (model)")
print(f"  - xgboost_metrics.json")
print(f"  - comparison.json")
print(f"  - xgboost_predicted_vs_actual.png")
print(f"  - xgboost_feature_importance.png")
print(f"")
print(f"HuggingFace repos:")
print(f"  Dataset:    https://huggingface.co/datasets/{HF_TABULAR}")
print(f"  XGBoost:    https://huggingface.co/{XGB_REPO}")
print(f"  QLoRA ONNX: https://huggingface.co/{QLORA_REPO}")
print(f"")
print(f"Total notebook time: {total_elapsed:.1f}s")
print("=" * 60)